# VitalGuard: Real-Time Wearable Stress & Affect Detection System

AAI-530 - Data Analytics and Internet of Things 

In [ ]:
# TODO: Remove before final submission, No longer needed in notebook
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added to path: {project_root}")


In [ ]:
import os
import zipfile
import pandas as pd
from typing import Any

from helpers.signal_processing import process_physiological_signals
from helpers.questionnaire_parsing import parse_event_timings, parse_questionnaire_responses
from helpers.file_io import load_subject_pickle
from helpers.data_merging import (
    create_merged_dataset, 
    save_merged_dataset, 
    upsample_to_64hz,
    downsample_to_64hz
)


## Import Data


### Extract zip file 

Its not necessary to run the code within this section every time. Once is fine

In [ ]:
# Define paths using your local structure
COMPRESSED_DATASET_PATH = 'DATA/COMPRESSED_DATASET/WESAD.ZIP'
EXTRACT_PATH = 'data/'

if os.path.exists(COMPRESSED_DATASET_PATH):
    with zipfile.ZipFile(COMPRESSED_DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"Extraction complete! Files are in: {EXTRACT_PATH}")
else:
    print("Zip file not found. Check the path again.")


### Load data & Process data

In [ ]:
# Define the dataset path after extraction
DATASET_PATH: str = 'data/WESAD'

SUBJECTS: list[str] = sorted([
    d for d in os.listdir(DATASET_PATH)
    if d.startswith("S")
])

# Array to hold dictionaries of processed data for each subject
processed_subjects_data: list[dict[str, Any]] = []

for subject in SUBJECTS:
    subject_pickle_path: str = os.path.join(DATASET_PATH, subject, f"{subject}.pkl")
    subject_quest_path: str = os.path.join(DATASET_PATH, subject, f"{subject}_quest.csv")
    questionnaire_df = pd.read_csv(subject_quest_path)

    if not os.path.exists(subject_pickle_path):
        continue

    print(f"Processing {subject}...")
    # subject_data_raw = pd.read_pickle(subject_pickle_path)
    subject_data_raw = load_subject_pickle(subject_pickle_path)

    # Process physiological signals and parse questionnaire data
    chest_df = process_physiological_signals(subject_data_raw['signal']['chest'])
    wrist_df = process_physiological_signals(subject_data_raw['signal']['wrist'])
    event_timings_df = parse_event_timings(questionnaire_df)
    questionnaire_responses_df = parse_questionnaire_responses(questionnaire_df)

    # Store processed data in a dictionary for the subject
    subject_processed_data: dict[str, Any] = { 
        'subject_id': subject,
        'chest': chest_df,
        'wrist': wrist_df,
        'event_timings': event_timings_df,
        'questionnaire_responses': questionnaire_responses_df
    }
    
    processed_subjects_data.append(subject_processed_data)


In [ ]:
#  Save the merged dataset for future use
MERGED_DATASET_PATH: str = 'data/merged_datasets'

# Merge sensor data with subject columns
merged_sensor_data: dict[str, Any] = create_merged_dataset(processed_subjects_data)

# Save the merged datasets
# save_merged_dataset(merged_sensor_data, MERGED_DATASET_PATH)

# Print summary of merged data
print("=" * 60)
print("Merged Data Summary:")
print("=" * 60)

print(f"Total subjects processed: {len(processed_subjects_data)}")
print(f"Event timings shape: {merged_sensor_data['event_timings'].shape}")
print(f"Questionnaire responses shape: {merged_sensor_data['questionnaire_responses'].shape}")

print("=" * 60)
print("Sensor data shapes:")
print("=" * 60)
for sensor_key in merged_sensor_data['sensors']:
    print(f"{sensor_key} data shape: {merged_sensor_data['sensors'][sensor_key].shape}")



### Align data

##### Downsample
Chest data frequency is 700Hz and needs downsampled to 64Hz

In [ ]:
aligned_chest_data = []

for subjects in processed_subjects_data:

    subject_id = subjects['subject_id']
    df_target_sub = subjects['wrist']  # BVP is the target
    high_freq_signal = subjects['chest']  # chest contains the high-freq signals

    # BVP is the target signal at 64Hz
    subject_bvp_df = df_target_sub['BVP'].copy()  # Keep only BVP column for target
    aligned_df = downsample_to_64hz(subject_id, subject_bvp_df, high_freq_signal)
    aligned_chest_data.append(aligned_df)

    print(f"Aligned data for {subject_id}: {aligned_df.shape[0]} rows, {aligned_df.shape[1]} columns")


#### Upsample
Various signals recorded from the wrist sensor are below 64Hz and need to be upsampled.

In [ ]:
aligned_wrist_data = []

for subjects in processed_subjects_data:

    subject_id = subjects['subject_id']
    df_target_sub = subjects['wrist']  # BVP is the target
    high_freq_signal = subjects['wrist']  # wrist contains the high-freq signals

    # BVP is the target signal at 64Hz
    subject_bvp_df = df_target_sub['BVP'].copy()  # Keep only BVP column for target
    aligned_df = upsample_to_64hz(subject_id, subject_bvp_df, high_freq_signal)
    aligned_wrist_data.append(aligned_df)

    print(f"Aligned data for {subject_id}: {aligned_df.shape[0]} rows, {aligned_df.shape[1]} columns")


In [ ]:
# Combine aligned wrist and chest data into a single DataFrame for each subject
combined_aligned_data = []

for wrist_df, chest_df in zip(aligned_wrist_data, aligned_chest_data):
    subject_id = wrist_df['subject_id'].iloc[0]  # Assuming subject_id is the same in both
    combined_df = pd.merge_asof(
        wrist_df.sort_values('time_sec'), 
        chest_df.sort_values('time_sec'), 
        on='time_sec', 
        direction='nearest', 
        suffixes=('_wrist', '_chest')
    )
    combined_aligned_data.append(combined_df)
    print(f"Combined aligned data for {subject_id}: {combined_df.shape[0]} rows, {combined_df.shape[1]} columns")


#### Add Labels